# Phase 4 — Baseline 1: Rule-Based Routing

Yen's K-shortest-paths algorithm over the maritime graph. Accepts a `weight_key` parameter so the same function can rank routes using either:
- `base_weight` (pure distance, ignores disruption)
- `disrupted_weight` (distance + disruption penalty from Phase 3)

This lets us show disrupted vs non-disrupted routing side by side for the demo.

In [1]:
import pickle
import networkx as nx
import pandas as pd
from itertools import islice

PROCESSED_DIR = "../data/processed"

# Load the disruption-tagged graph (has both base_weight and disrupted_weight on every edge)
with open(f"{PROCESSED_DIR}/maritime_graph_disrupted.gpickle", "rb") as f:
    G = pickle.load(f)

print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Quick sanity check: confirm both weight attributes exist on edges
sample_edge = list(G.edges(data=True))[0]
print("Sample edge attrs:", sample_edge[2])

Graph loaded: 1543 nodes, 9462 edges
Sample edge attrs: {'distance_nm': 175.2965283650975, 'base_weight': 175.2965283650975, 'disrupted_weight': 175.2965283650975, 'disruption_severity_edge': 0.0}


In [2]:
def port_lookup_by_name(G, name_substring):
    """Helper: find port_id(s) by partial name match, for picking origin/destination during testing."""
    matches = [
        (node_id, attrs['port_name'], attrs['country'])
        for node_id, attrs in G.nodes(data=True)
        if name_substring.lower() in attrs['port_name'].lower()
    ]
    return matches

# Example usage
port_lookup_by_name(G, "singapore")

[(50000, 'Keppel - (East Singapore)', 'Singapore')]

In [3]:
def k_shortest_paths(G, origin_id, dest_id, k=5, weight_key='disrupted_weight'):
    """
    Returns the K shortest paths between origin_id and dest_id using the specified
    edge weight attribute ('base_weight' or 'disrupted_weight').

    Uses NetworkX's shortest_simple_paths generator (Yen's algorithm equivalent),
    which yields paths in increasing order of weight.
    """
    if origin_id not in G:
        raise ValueError(f"Origin port_id {origin_id} not found in graph")
    if dest_id not in G:
        raise ValueError(f"Destination port_id {dest_id} not found in graph")
    if not nx.has_path(G, origin_id, dest_id):
        raise ValueError(f"No path exists between {origin_id} and {dest_id}")

    paths_generator = nx.shortest_simple_paths(G, origin_id, dest_id, weight=weight_key)

    results = []
    for path in islice(paths_generator, k):
        total_weight = sum(
            G[path[i]][path[i+1]][weight_key] for i in range(len(path) - 1)
        )
        total_distance_nm = sum(
            G[path[i]][path[i+1]]['distance_nm'] for i in range(len(path) - 1)
        )
        max_edge_disruption = max(
            (G[path[i]][path[i+1]].get('disruption_severity_edge', 0.0) for i in range(len(path) - 1)),
            default=0.0
        )
        results.append({
            'path': path,
            'path_port_names': [G.nodes[p]['port_name'] for p in path],
            'num_hops': len(path) - 1,
            'total_weight': round(total_weight, 2),
            'total_distance_nm': round(total_distance_nm, 2),
            'max_disruption_exposure': round(max_edge_disruption, 3),
            'weight_key_used': weight_key
        })

    return results

In [4]:
def print_routes(routes, title="Routes"):
    print(f"\n=== {title} ===")
    for i, r in enumerate(routes, 1):
        route_str = " -> ".join(r['path_port_names'])
        print(f"{i}. [{r['num_hops']} hops, {r['total_distance_nm']} nm, "
              f"weight={r['total_weight']}, disruption_exposure={r['max_disruption_exposure']}]")
        print(f"   {route_str}")

## Test: pick an origin-destination pair that passes near a disrupted region
Using the Phase 3 fallback scenario (Red Sea / Suez), a good test pair is a port on the Europe side vs a port on the Asia side — a real voyage would normally transit the Red Sea/Suez corridor.

In [5]:
# Find candidate origin/destination near Europe and Asia
print("Europe candidates:", port_lookup_by_name(G, "rotterdam"))
print("Asia candidates:", port_lookup_by_name(G, "singapore"))

Europe candidates: [(31140, 'Rotterdam', 'Netherlands')]
Asia candidates: [(50000, 'Keppel - (East Singapore)', 'Singapore')]


In [6]:
# --- Set these based on the lookup results above ---
ORIGIN_ID = 31140   # <-- fill in from port_lookup_by_name output
DEST_ID = 50000     # <-- fill in from port_lookup_by_name output

assert ORIGIN_ID is not None and DEST_ID is not None, "Set ORIGIN_ID and DEST_ID from the lookup above before running"

routes_disrupted = k_shortest_paths(G, ORIGIN_ID, DEST_ID, k=5, weight_key='disrupted_weight')
print_routes(routes_disrupted, title="Disrupted routing (avoids penalized edges)")

routes_baseline = k_shortest_paths(G, ORIGIN_ID, DEST_ID, k=5, weight_key='base_weight')
print_routes(routes_baseline, title="Non-disrupted routing (pure shortest distance)")


=== Disrupted routing (avoids penalized edges) ===
1. [41 hops, 14322.79 nm, weight=14322.79, disruption_exposure=0.0]
   Rotterdam -> Zeebrugge -> Oostende -> Boulogne-Sur-Mer -> Fecamp -> Port De Caen -> Saint-Malo -> Nantes -> Le Verdon -> Santander -> Aviles -> Aveiro -> Lagos -> Safi -> Agadir -> Nouakchott -> Conakry -> Tema -> Pennington Oil Terminal -> Port Owendo -> Takula Terminal -> Palanca Terminal -> Walvis Bay -> Luderitz Bay -> Richards Bay -> Beira -> Dar Es Salaam -> Muqdisho -> Boosaaso -> Mina Raysut -> Mina Al Fahl -> Gwadar -> Mundra -> Marmagao -> Kattupalli Port -> Gopalpur -> Bassein -> Mergui -> Phuket -> Pulau Pinang -> Melaka -> Keppel - (East Singapore)
2. [42 hops, 14322.81 nm, weight=14322.81, disruption_exposure=0.0]
   Rotterdam -> Zeebrugge -> Oostende -> Boulogne-Sur-Mer -> Fecamp -> Port De Caen -> Saint-Malo -> Nantes -> La Pallice -> Le Verdon -> Santander -> Aviles -> Aveiro -> Lagos -> Safi -> Agadir -> Nouakchott -> Conakry -> Tema -> Pennington

## Side-by-side comparison
If disruption is working correctly, the top disrupted-route recommendation should differ from the top non-disrupted route (or have higher distance but lower disruption exposure), for pairs whose shortest path passes near a disrupted zone.

In [7]:
comparison = pd.DataFrame([
    {
        'rank': i+1,
        'disrupted_route': " -> ".join(routes_disrupted[i]['path_port_names']) if i < len(routes_disrupted) else None,
        'disrupted_distance_nm': routes_disrupted[i]['total_distance_nm'] if i < len(routes_disrupted) else None,
        'disrupted_exposure': routes_disrupted[i]['max_disruption_exposure'] if i < len(routes_disrupted) else None,
        'baseline_route': " -> ".join(routes_baseline[i]['path_port_names']) if i < len(routes_baseline) else None,
        'baseline_distance_nm': routes_baseline[i]['total_distance_nm'] if i < len(routes_baseline) else None,
    }
    for i in range(max(len(routes_disrupted), len(routes_baseline)))
])
comparison

,rank,disrupted_route,disrupted_distance_nm,disrupted_exposure,baseline_route,baseline_distance_nm
0,1,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,14322.79,0.0,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8569.80
1,2,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,14322.81,0.0,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8569.81
2,3,Rotterdam -> Zeebrugge -> Dunkerque Port Est -...,14322.81,0.0,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8569.81
3,4,Rotterdam -> Zeebrugge -> Oostende -> Dunkerqu...,14322.81,0.0,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8569.82
4,5,Rotterdam -> Zeebrugge -> Dunkerque Port Est -...,14322.83,0.0,Rotterdam -> Zeebrugge -> Oostende -> Boulogne...,8569.83


In [8]:
# --- Save Baseline 1 as a reusable module for the API (Phase 6) ---
# This cell content will be converted into src/ranking/baseline.py

print("Baseline 1 functions ready: k_shortest_paths(), port_lookup_by_name()")
print("These will be moved into src/ranking/baseline.py for reuse in the API.")

Baseline 1 functions ready: k_shortest_paths(), port_lookup_by_name()
These will be moved into src/ranking/baseline.py for reuse in the API.
